Given 2 strings S1 and S2, conpute the minimum number of operations needed to convert S1 into S2. We can repalce a character, delete a charater or insert a character in S1.  
Example :- Input : S1 : "horse" -> S2 : "ros" -> Output : 3 operations.

## Recursive Implementation
First let us do a recursive implementation. Let i and j be indexes into S1 and S2. At each recursion if S1{i} is not equal to S2{j}, then we have 3 options replace, delete or insert. We have to choose the option with the least operations in these 3 and move forward. 

```C++
int min_operations(string &word1, int i, string &word2, int j)
{
    //If word1 has exhausted, then we have to insert the remaining elements in word2
    if(i == word1.size())
    {
        return word2.size() - j;
    }

    //If word2 has exhausted, then we have to delete the remaining elements in word1
    if(j == word2.size())
    {
        return word1.size() - i;
    }

    //Check the characters and do the optimal operations
    //If both characters are same
    if(word1[i] == word2[j])
    {
        return min_operations(word1, i+1, word2, j+1);
    }
    else
    {
        //1 for doing one of the 3 operations
        return 1 + min({
            min_operations(word1, i+1, word2, j+1), //Replace, move both i and j as the both characters are same now
            min_operations(word1, i+1, word2, j), //Delete, only move i as that character is deleted
            min_operations(word1, i, word2, j+1)  //Insert, move only j as S[j] is inserted and matches
        });
    }
}


int minDistance(string word1, string word2) 
{
    return min_operations(word1, 0, word2, 0);
}
``` 
We have 3 choices for each character, so the time complexity is O(3<sup>N</sup>). Space complexity is O(N) for the max stack space(full recursive branch).

## Top Down Memoization
Is this solving same subproblems repeatedly? Yes, after making different choices(replace, delete or insert), we might end up with the same sub problem(word) on different recursive paths.  
In order to identify a structure to save the answers of the subproblems, we need to identify the parameters that are continuously changing in the recursive function, which are i and j for this recursive function, we can use a 2D array of size M(S1 size)+1 x N(S2 size)+1. 
So solving this with top down memoization.  

```C++
vector<vector<int>> dp;
int min_operations(string &word1, int i, string &word2, int j)
{
    //If word1 has exhausted, then we have to insert the remaining elements in word2
    if(i == word1.size())
    {
        return word2.size() - j;
    }

    //If word2 has exhausted, then we have to delete the remaining elements in word1
    if(j == word2.size())
    {
        return word1.size() - i;
    }

    if(dp[i][j] != -1)
    {
        return dp[i][j];
    }

    //Check the characters and do the optimal operations
    //If both characters are same
    if(word1[i] == word2[j])
    {
        dp[i][j] = min_operations(word1, i+1, word2, j+1);
    }
    else
    {
        //1 for doing one of the 3 operations
        dp[i][j] = 1 + min({
            min_operations(word1, i+1, word2, j+1), //Replace, move both i and j as the both characters are same now
            min_operations(word1, i+1, word2, j), //Delete, only move i as that character is deleted
            min_operations(word1, i, word2, j+1)  //Insert, move only j as S[j] is inserted and matches
        });
    }

    return dp[i][j];
}

int minDistance(string word1, string word2) 
{
    for(int i = 0; i <= word1.size(); i++)
    {
        vector<int> col;
        col.resize(word2.size() + 1, -1);

        dp.push_back(col);
    }
    return min_operations(word1, 0, word2, 0);

}
```
Time complexity is O(M * N). Space complexity will be O(M * N) for DP array and O(M + N) for the stack.

## Bottom up Iterative
How do we convert this into a bottom up idea? We can do the following pattern
* What kind of data structure do we need : 2D array from the memoization solution
* What is the size : m+1, n+1, from the memoization solution
* Fill the data structure with the base conditions : mth row and nth col, from the recursive solution
* What direction should we populate the data structure : Known to unknown solutions(base conditions to full solution), m to 0, n to 0(row by row form the last).
* Copy the recursive logic.

```C++
vector<vector<int>> dp;
int minDistance(string word1, string word2) 
{

    int m = word1.size();
    int n = word2.size();

    //Dp array
    vector<vector<int>> dp;
    for(int i = 0; i <= m; i++)
    {
        vector<int> col;
        col.resize(n+1, -1);

        dp.push_back(col);
    }

    //Fill the base conditions, same conditions from recursion
    //mth row and nth column
    for(int j = 0; j <= n; j++)
    {
        dp[m][j] = n - j;
    }
    for(int i = 0; i <= m; i++)
    {
        dp[i][n] = m - i;
    }

    //Iteratively populate the rest of the grid in the reverse direction
    //Copy the recursive logic
    for(int i = m-1; i >= 0; i--)
    {
        for(int j = n-1; j >= 0; j--)
        {
            if(word1[i] == word2[j])
            {
                dp[i][j] = dp[i+1][j+1];
            }
            else
            {
                //1 for doing one of the 3 operations
                dp[i][j] = 1 + min({dp[i+1][j+1], dp[i+1][j], dp[i][j+1]});
            }
        }
    }

    return dp[0][0];
}
```
Time complexity will be same. Space complexity will be only O(M * N) as there is no call stack.